# Zero-Shot Text Classification

Classifies each row's text into user-defined categories without any training data, using `facebook/bart-large-mnli` via the HuggingFace Inference API.

Single-label mode adds one string column. Multi-label mode adds a `#multi` pipe-separated column.

A free HuggingFace token improves rate limits but is not required. Get one at https://huggingface.co/settings/tokens

In [ ]:
# -- Colab bootstrap (no-op on Binder / JupyterHub) ------------------------
import os, sys
if "google.colab" in sys.modules:
    import subprocess
    if os.path.isdir("/tmp/suave-nb/.git"):
        subprocess.run(["git","-C","/tmp/suave-nb","fetch","--depth=1","origin","main"],
                       capture_output=True)
        subprocess.run(["git","-C","/tmp/suave-nb","reset","--hard","origin/main"],
                       capture_output=True)
    else:
        _r = subprocess.run(
            ["git","clone","--depth=1",
             "https://github.com/izaslavsky/suave-notebooks.git","/tmp/suave-nb"],
            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Colab only: mount Google Drive to load session credentials ─────────────
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        from IPython.display import HTML as _HTML
        display(_HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE parameters found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open this notebook via SuAVEDispatch from your survey in SuAVE.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
else:
    # Colab: try Drive/cache silently; credentials cell below handles fallback
    _p = su.load_params(_silent=True)


In [ ]:
# -- Colab: verify session loaded from Drive/cache --------------------------
from IPython.display import HTML
if su.ENV.colab:
    if _p is None:
        display(HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE session found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open SuAVEDispatch from the correct survey in SuAVE, then re-open this notebook.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
    display(HTML(
        '<div style="font-size:12px;padding:8px 12px;border-radius:4px;margin:3px 0;'
        'background:#e8f5e9;border:1px solid #81c784">'
        'Session loaded &mdash; survey <b>' + _p.get('survey', '?') + '</b>'
        ', user <b>' + _p.get('user', '?') + '</b>.'
        '<br><span style="color:#666;font-size:11px">'
        'Wrong survey? Go back to SuAVE, open the correct survey, and click Jupyter again.'
        '</span></div>'))
else:
    su._skipped('Colab only')


In [ ]:
# ── Variables used throughout this notebook ────────────────────────────────
if _p is None:
    from IPython.display import HTML as _HTML
    display(_HTML(
        '<div style="background:#dc3545;color:white;padding:14px 16px;'
        'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
        '&#9888; Parameters not loaded.'
        '<br><span style="font-size:12px;font-weight:normal">'
        'Navigate to your survey in SuAVE and relaunch this notebook to load parameters.'
        '</span></div>'))
    class _Stop(Exception):
        def _render_traceback_(self): return []
    raise _Stop()
user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np

In [ ]:
# Optional HuggingFace token — improves rate limits, not required
# Get a free token at https://huggingface.co/settings/tokens
HF_TOKEN = ''   # @param {type:"string"}

client = su.get_hf_client(HF_TOKEN)

## 1. Load survey data and configure classification

In [ ]:
df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

text_cols = [c for c in df.columns if '#number' not in c and '#img' not in c
             and '#hidden' not in c and '#date' not in c]

col_selector = widgets.Dropdown(
    options=text_cols, description='Text column:',
    layout=widgets.Layout(width='60%')
)
labels_input = widgets.Textarea(
    value='Technology, Health, Politics, Environment, Economy',
    description='Labels (comma-separated):',
    layout=widgets.Layout(width='60%', height='60px')
)
mode_selector = widgets.RadioButtons(
    options=['Single-label (best match)', 'Multi-label (all above threshold)'],
    description='Mode:', layout=widgets.Layout(width='60%')
)
threshold_slider = widgets.FloatSlider(
    value=0.5, min=0.1, max=0.95, step=0.05,
    description='Threshold:', continuous_update=False,
    layout=widgets.Layout(width='60%')
)
output_col_input = widgets.Text(
    value='Category', description='Output column:',
    layout=widgets.Layout(width='60%')
)

def _toggle_threshold(change):
    threshold_slider.disabled = 'Single' in change['new']

mode_selector.observe(_toggle_threshold, names='value')
threshold_slider.disabled = True

display(col_selector, labels_input, mode_selector, threshold_slider, output_col_input)

## 2. Run zero-shot classification

In [ ]:
MODEL = 'facebook/bart-large-mnli'

col        = col_selector.value
candidates = [l.strip() for l in labels_input.value.split(',') if l.strip()]
multi      = 'Multi' in mode_selector.value
threshold  = threshold_slider.value
out_col    = output_col_input.value.strip() or 'Category'
if multi:
    out_col = out_col.rstrip('#multi') + '#multi'

printmd(f"**Classifying `{col}` into: {candidates}**")

from tqdm.auto import tqdm

results = []
for text in tqdm(df[col], desc='Classifying'):
    if not text or pd.isna(text) or str(text).strip() == '':
        results.append(None)
        continue
    try:
        resp = client.zero_shot_classification(
            str(text)[:512],
            labels=candidates,
            multi_label=multi,
            model=MODEL
        )
        label_scores = dict(zip(resp.labels, resp.scores))
        if multi:
            chosen = [l for l, s in label_scores.items() if s >= threshold]
            results.append('|'.join(chosen) if chosen else None)
        else:
            results.append(resp.labels[0])
    except Exception as e:
        results.append(None)

df[out_col] = results

printmd(f'**Done. New column: `{out_col}`. Sample:**')
display(df[[col, out_col]].dropna().head(10))

## 3. Review distribution

In [ ]:
import matplotlib.pyplot as plt

# For multi-label, explode pipe-separated values
if '#multi' in out_col:
    counts = (df[out_col].dropna()
              .str.split('|').explode()
              .value_counts())
else:
    counts = df[out_col].value_counts()

counts.plot.barh(figsize=(8, 4))
plt.title(f'Distribution of {out_col}')
plt.tight_layout()
plt.show()

## 4. Save and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
#Input survey name
survey_name = input('Enter survey name: ')
printmd('**Survey Name:** ' + survey_name)


In [ ]:
suaveint.create_survey(survey_url, new_file, survey_name, dzc_file, user, csv_file, view, views)
